In [1]:
import os

def list_markdown_files(md_dir):
    return [
        os.path.join(md_dir, f)
        for f in os.listdir(md_dir)
        if f.endswith(".md")
    ]


In [2]:
import re

def split_by_h2(md_text):
    pattern = r"(?=^##\s+)"
    sections = re.split(pattern, md_text, flags=re.MULTILINE)

    results = []
    for sec in sections:
        lines = sec.strip().splitlines()
        if not lines or not lines[0].startswith("##"):
            continue

        title = lines[0].replace("##", "").strip()
        content = "\n".join(lines[1:]).strip()

        results.append((title, content))

    return results


In [3]:
import os
from langchain_core.documents import Document

MD_DIR = "dataMd"
all_chunks = []

for md_file in list_markdown_files(MD_DIR):
    with open(md_file, "r", encoding="utf-8") as f:
        md_text = f.read()

    sections = split_by_h2(md_text)

    for title, content in sections:
        if not content.strip():
            continue

        doc = Document(
            page_content=content,
            metadata={
                "source_file": os.path.basename(md_file),
                "section_title": title
            }
        )

        all_chunks.append(doc)

print(f"Total chunks générés : {len(all_chunks)}")


Total chunks générés : 1009


In [4]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

PERSIST_DIR = "./chroma_db"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR
)
vector_store.persist()


C:\Users\marti\AppData\Local\Temp\ipykernel_26024\849729678.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
C:\Users\marti\AppData\Local\Temp\ipykernel_26024\849729678.py:15: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()


In [6]:
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage
from typing import List, TypedDict
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Charger le vector store ChromaDB
vector_store = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

# Définir le LLM (Ollama)
llm = Ollama(model="ministral-3", base_url="http://localhost:11434")

# Créer le retriever
retriever = vector_store.as_retriever()

# Définir le prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant utile. Utilise uniquement le contexte suivant pour répondre à la question. Si tu ne connais pas la réponse, dis simplement que tu ne sais pas."),
    ("human", "Contexte: {context}\n\nQuestion: {question}")
])

# Créer la chaîne de traitement
output_parser = StrOutputParser()
chain = prompt | llm | output_parser

# Définir l'état
class State(TypedDict):
    messages: List[HumanMessage]

# Node pour répondre à la question
def answer_node(state: State):
    last_msg = state["messages"][-1].content
    # Récupérer les documents pertinents
    docs = retriever.invoke(last_msg)
    context = "\n".join([doc.page_content for doc in docs])
    # Générer la réponse
    response = chain.invoke({"context": context, "question": last_msg})
    return {"messages": state["messages"] + [AIMessage(content=response)]}

# Créer le graphe LangGraph
graph = StateGraph(State)
graph.add_node("answer", answer_node)
graph.set_entry_point("answer")
graph.add_edge("answer", END)
app = graph.compile()

# Exemple d'utilisation (simule une question venue d'Otoroshi)
Question = "Quels sont les risques majeurs concernant Le Tallud?"
result = app.invoke({"messages": [HumanMessage(content=Question)]})
print("Réponse :", result["messages"][-1].content)


Réponse : D'après le contexte fourni, voici les risques majeurs **potentiellement** concernants **Le Tallud** (commune située dans le département de la Vienne) :

1. **Risque inondation** (mentionné comme un risque à prendre en compte dans l’aménagement du territoire, bien que la Vienne soit globalement peu exposée).

Aucune autre précision n’est donnée dans le contexte pour les 17 autres risques majeurs. Si tu veux des détails plus précis, il faudrait consulter les documents officiels du département ou de la commune.


In [7]:
test = ["Quels sont les risques naturels et technologiques majeurs recensés dans le département de la Vienne selon le DDRM ?",
"Quelles sont les principales rivières concernées par le risque inondation dans la Vienne et quels types d'inondations peuvent survenir ?",
"Comment est classé le département de la Vienne en termes de zonage sismique ?",
"Quelles sont les mesures de pré-distribution de comprimés d'iode prévues autour de la centrale nucléaire de Civaux ?",
"Quels sont les établissements classés SEVESO seuil haut dans le département de la Vienne et où sont-ils situés ?",
"De quoi se compose le Signal National d'Alerte et comment reconnaît-on le signal de fin d'alerte ?",
"Quelles sont les consignes avant, pendant et après une inondation?",
"Quels sont les risques majeurs concernant Le Tallud?"]

for question in test:
    print(question)
    result = app.invoke({"messages": [HumanMessage(content=question)]})
    print("Réponse :", result["messages"][-1].content)

Quels sont les risques naturels et technologiques majeurs recensés dans le département de la Vienne selon le DDRM ?
Réponse : D'après le contexte fourni, je ne peux pas répondre directement aux risques spécifiques au **département de la Vienne**, car celui-ci concerne uniquement le **Deux-Sèvres** (mentionné dans les liens et exemples donnés : `www.deux-sevres.gouv.fr`).

Cependant, voici les **éléments généraux** que le **DDRM** (Dossier Départemental des Risques Majeurs) doit inclure pour tout département, en s'appuyant sur les références données (notamment pour la Savoie, mais transposables) :

### **Risques naturels majeurs** (exemples courants en France) :
1. **Inondations** (pluviales, crues torrentielles, remontées de nappe).
2. **Mouvements de terrain** (glissements de terrain, effondrements, retrait-gonflement des argiles).
3. **Séismes** (faible à modéré selon les zones).
4. **Tempêtes et vents violents** (cyclones, orages).
5. **Sécheresses et canicules** (impacts sur les re